# 08. The cap-weight check: building the index the extension will track

**Terms used in this notebook.**

| Term | Meaning |
|---|---|
| Risk-free rate | the return on a one-month US Treasury bill, the closest thing to a return with no risk |
| Excess return | a return minus the risk-free rate over the same month; French's Mkt-RF is the excess return of the whole US stock market |
| Market capitalisation | the number of a firm's shares times their price, the firm's size in money; market equity is the same quantity |
| Value-weighted return | the return of a group of firms in which each firm counts in proportion to its market capitalisation at the start of the month, so a firm twice as large counts twice as much |
| Cap-weighted | weighted by market capitalisation; a cap-weighted index of industries gives each industry the share of the total that its firms' market capitalisation amounts to |
| Benchmark | in the extension, the index the portfolio tracks |
| Tracking error | the standard deviation of the monthly difference between two return series, times the square root of 12, so that it reads in return units per year; a tracking error of 0.50% a year means the two series differ by about 0.14% in a typical month (0.50% divided by the square root of 12) |
| Basis point | one hundredth of a percentage point, so 50 basis points is 0.50% |
| Look-ahead | using, for month t, a number that was not known when month t began; a weight set with the end-of-month price is look-ahead |
| Vintage | the version of Ken French's files on the download date, named by the release of the CRSP database they were built from (202608 is the August 2026 release) |
| Provenance | the record of what was downloaded, when, with its checksum and vintage |
| SIC code | the Standard Industrial Classification, a four-digit number that names a firm's line of business; French assigns each firm to one of the 49 industries by its SIC code at the end of June |
| Firm-level data | one row per firm per month (price, shares, return), as opposed to French's files, which carry one number per portfolio per month |

**What this notebook does.** The extension (notebooks 09 to 13) asks the paper's question of an index-tracking mandate on French's 49 industry portfolios: a portfolio that must stay close to an index of those industries. That index has to be built first, because French publishes the 49 industries' returns and the market's return, and no index of the 49. French's 49-industry file carries, for every industry and month, the number of firms and the average firm size (market capitalisation in millions of dollars). Their product is the industry's total market capitalisation, and dividing by the sum over the 49 gives each industry's share of the whole: its cap weight. The cap-weighted combination of the 49 industry returns should then equal, or nearly equal, the value-weighted return of all the firms in the 49 portfolios, and that is close to French's market return. This notebook builds the weights, computes the combination, and measures its tracking error against the market, Mkt-RF plus RF, from 1969-07 (the first month in which all 49 industries exist, asserted in notebook 01) to the last month the vintage carries.

**The check, fixed in `constants.py` on 10 September 2026, before any extension code existed.** If the tracking error is at most 50 basis points a year (`CAPW_TE_MAX_ANNUAL`), the derived weights are the benchmark of the extension, both its return series and its industry weights. If not, the benchmark return is the market series, and the question of which industry weights the constraints refer to goes back to the design before notebook 09 is written.

**Why the timing of the weights matters, and how it is checked.** A value-weighted return for month t weights each firm by its market capitalisation at the start of the month, because an investor who holds the market buys at the start of the month and the weights she holds are those prices times shares. French's firm count and average size for month t are measured at the same time, the start of month t, so the product for month t is the weight that goes with month t's return, and it uses nothing that was unknown when the month began. The other reading, that French's numbers describe the end of the month, would make the same-month weight look-ahead. The two readings give different combinations, so both are run: the same-month weights, which are the design (`CAPW_WEIGHT_TIMING`), and the previous-month weights (the count and size of month t-1 applied to month t's return). Under the design's reading, the same-month combination must track the market more closely. If it does not, the reading is wrong, the notebook stops, and the extension is not built on a benchmark with look-ahead.

**What I expect to see, written before the run (`constants.py`, 1 October 2026).** First, count first: from 1969-07 to the last month, every industry has a return, a firm count and an average size in every month, and the three blocks share the same months. Second, the same-month weights track the market more closely than the previous-month weights. Third, the gate: the same-month combination's tracking error is at most 50 basis points a year. Fourth, reported with no pass or fail: the annualised mean difference between the combination and the market, their correlation, and the largest and smallest weights with their industries at the first month, at 2004-11 and at the last month. Running time: seconds.

## Modules

`constants` and `french_loader` are those of notebook 01. `benchmark` is new: the weights, the combination and the tracking error, each a short function with a hand-computed test in `tests/test_benchmark.py`.

In [ ]:
import os
os.makedirs("src/bp", exist_ok=True)
open("src/bp/__init__.py", "w").close()

In [ ]:
%%writefile src/bp/constants.py
"""
Pre-committed numbers for the study.

Every band, gate, sample boundary and design parameter is defined here, one
per line, with its source beside it. A notebook may read these names; it may
not introduce numbers of its own. The replication bands (section 3) and the
design of the extension (section 5) were fixed on 10 September 2026, before any
of the rules' code was written. The expectations each notebook tests are
stated in that notebook's introduction and were written before its code; where
a level was changed after a run, the comment beside it says so, with the date
and the reason, so that a reader can judge whether the change was a
specification settled by the paper's own text or a number tuned to the results.

DGU throughout means DeMiguel, Garlappi and Uppal (2009), "Optimal Versus
Naive Diversification: How Inefficient is the 1/N Portfolio Strategy?",
Review of Financial Studies 22(5), 1915-1953. Page numbers are the journal's.
"""

# ---------------------------------------------------------------------------
# 1. Data: Ken French's library, downloaded at run time, never committed
# ---------------------------------------------------------------------------

FRENCH_BASE_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"

# Every file the study reads. The vintage is whatever the library serves on the
# download date; the loader records the date, the SHA-256 of the zip and the
# CRSP vintage line printed at the top of each CSV.
FRENCH_FILES = {
    "factors3":   "F-F_Research_Data_Factors_CSV.zip",        # Mkt-RF, SMB, HML, RF, monthly from 1926-07
    "momentum":   "F-F_Momentum_Factor_CSV.zip",              # Mom, monthly from 1927-01
    "factors5":   "F-F_Research_Data_5_Factors_2x3_CSV.zip",  # Mkt-RF, SMB, HML, RMW, CMA, RF from 1963-07
    "ind10":      "10_Industry_Portfolios_CSV.zip",           # DGU "Industry" dataset
    "ind49":      "49_Industry_Portfolios_CSV.zip",           # the extension universe; carries firm counts and average size
    "size_bm25":  "25_Portfolios_5x5_CSV.zip",                # DGU "FF-1-factor" and "FF-4-factor" datasets
    # Daily files, for the extension's covariance estimators (section 5).
    "ind49_daily":    "49_Industry_Portfolios_daily_CSV.zip",        # value- and equal-weighted daily returns from 1926-07-01
    "factors3_daily": "F-F_Research_Data_Factors_daily_CSV.zip",     # Mkt-RF, SMB, HML, RF daily from 1926-07-01
    "factors5_daily": "F-F_Research_Data_5_Factors_2x3_daily_CSV.zip", # Mkt-RF, SMB, HML, RMW, CMA, RF daily from 1963-07-01
    "momentum_daily": "F-F_Momentum_Factor_daily_CSV.zip",           # Mom daily from 1926-11-03
}

FRENCH_MISSING_CODES = (-99.99, -999.0)   # French's own convention, stated at the top of every portfolio file
FRENCH_RETURNS_ARE_PERCENT = True         # the loader divides by 100 once, at load time, and nowhere else

# ---------------------------------------------------------------------------
# 2. The DGU replication
# ---------------------------------------------------------------------------

DGU_SAMPLE_START = "1963-07"   # DGU Table 2, p. 1918, for every French-sourced dataset
DGU_SAMPLE_END = "2004-11"     # DGU Table 2, p. 1918
DGU_ESTIMATION_WINDOW = 120    # months; DGU section 2, p. 1927. M = 60 is in DGU's appendix and is out of scope.
DGU_RISK_AVERSION = 1.0        # gamma in the CEQ, DGU p. 1929: "the results we report are for the case of gamma = 1"
DGU_TCOST = 0.0050             # proportional cost per unit of one-way turnover; DGU p. 1929: 50 basis points, after Balduzzi and Lynch (1999)
TCOST_SENSITIVITY = 0.0100     # the one cost sensitivity reported: double the base cost
DGU_GMINC_LOWER_BOUND_FRACTION = 0.5   # g-min-c imposes w >= a * 1 with a = 1/(2N); DGU p. 1926
DGU_BS_COV_DOF = "M - N - 2"   # the Bayes-Stein covariance divides by M - N - 2; DGU equation (5), p. 1923

# Settled 22 September 2026, a specification the paper leaves ambiguous. DGU's
# Lagrangian (8), p. 1925, shows the short-sale constraint alone; footnote 22,
# p. 1934, says the constrained rules produce "corner solutions with all wealth
# invested in a single asset", which only happens when the budget constraint
# 1'w = 1 sits inside the optimisation with gamma = 1. With the budget inside,
# mv-c on MKT/SMB/HML gives a Sharpe ratio of 0.1093 against the published
# 0.1084 and a corner solution in 257 of 377 months; without it, 0.2382 and no
# corner. The reading that reproduces the paper's tables is used; the literal
# reading is kept as rules.mv_c_cone for comparison (notebook 03).
DGU_CONSTRAINED_BUDGET_INSIDE = True

# The rules replicated: those whose only inputs are the window's sample moments
# (and, for Bayes-Stein, Jorion's prior built from them), plus the value-weighted
# market, whose Sharpe ratio is the same 0.1138 in every French-sourced column
# of Table 3 and so checks that the market series and the sample match the paper's.
DGU_RULES = ("ew", "mv", "bs", "min", "vw", "mv-c", "bs-c", "min-c", "g-min-c")
DGU_IN_SAMPLE_ROW = "mv (in sample)"   # DGU Table 3 row 2: mean-variance with M = T, the upper bound estimation error costs

# Excluded, each with the criterion. They are not built.
DGU_RULES_EXCLUDED = {
    "Bayesian diffuse-prior": "not reported by DGU (Table 1 note, p. 1917)",
    "dm": "Bayesian Data-and-Model (Pastor 2000); needs an asset-pricing prior, outside the moment-based set",
    "mp": "MacKinlay and Pastor (2000) missing-factor model; needs a maximum-likelihood factor structure, outside the moment-based set",
    "mv-min": "Kan and Zhou (2007) three-fund mixture of two rules already built; not part of the pre-specified set",
    "ew-min": "mixture of 1/N and minimum variance; not part of the pre-specified set",
    "multi-prior": "not reported by DGU (Table 1 note, p. 1917)",
}

# The datasets replicated: the four DGU built from French's site (DGU Table 2, p. 1918; Appendix A, pp. 1948-1949).
# Each is monthly excess returns over the T-bill rate from French's factor file.
DGU_DATASETS = {
    "Industry":     {"n_risky": 10, "factors": ("Mkt-RF",), "N": 11, "source": "ind10 + factors3"},
    "MKT/SMB/HML":  {"n_risky": 0,  "factors": ("Mkt-RF", "SMB", "HML"), "N": 3, "source": "factors3"},
    "FF-1-factor":  {"n_risky": 20, "factors": ("Mkt-RF",), "N": 21, "source": "size_bm25 + factors3"},
    "FF-4-factor":  {"n_risky": 20, "factors": ("Mkt-RF", "SMB", "HML", "Mom"), "N": 24, "source": "size_bm25 + factors3 + momentum"},
}
DGU_DATASETS_EXCLUDED = {
    "S&P Sectors": "Roberto Wessels' GICS sector data, not on French's site (DGU Appendix A.1)",
    "International": "MSCI country indices, not on French's site (DGU Appendix A.3)",
    "FF-3-factor": "DGU do not report it: 'very similar to FF-1-factor' (Table 3 note, p. 1931)",
}
DGU_EXCLUDE_LARGEST_SIZE_QUINTILE = True   # DGU footnote 24, p. 1949: the five largest-size portfolios of the 25 are dropped, leaving 20

# ---------------------------------------------------------------------------
# 3. Bands for the replication, written before the code ran
# ---------------------------------------------------------------------------

BAND_SHARPE_ABS = 0.03          # out-of-sample Sharpe ratio within 0.03 of DGU Table 3, per rule and dataset
CHECK_TOP3_RANKING = True        # the ranking of the top three rules within each dataset is preserved
BAND_TURNOVER_REL = 0.25        # turnover within 25% relative of DGU Table 5

# DGU Table 5 reports the unconstrained mean-variance rule's turnover at 606,594
# times that of 1/N on the Industry dataset and 10,466 times on FF-1-factor.
# Those figures are dominated by the months in which the sample covariance is
# near-singular and move by orders of magnitude when French revises a single
# return, so the band applies only where DGU's published relative turnover is
# below the level here; above it, the replicated figure is reported beside the
# published one with no pass or fail. Fixed 10 September 2026.
BAND_TURNOVER_APPLIES_BELOW_RELATIVE = 100.0

# Changed 13 September 2026, after notebook 02 had run, on its evidence: the
# out-of-sample Sharpe band applies only to rules whose published relative
# turnover (DGU Table 5, panel A) is below the same level, and the in-sample
# mean-variance row carries no verdict. Evidence: on the Industry, FF-1 and FF-4
# datasets the unconstrained mean-variance rule moved by 0.12 to 0.25 in Sharpe
# ratio under ten basis points of noise, with the published figure inside the
# range, while the 1/N rows matched to the third decimal. Before the change the
# band applied to every out-of-sample row. The level is the paper's own number,
# already used by the turnover band.
BAND_SHARPE_APPLIES_BELOW_RELATIVE = BAND_TURNOVER_APPLIES_BELOW_RELATIVE
BAND_SHARPE_DECISION_DATE = "2026-09-13"

# Sharpe ratios are compared on the same sample, 1963-07 to 2004-11, 120-month
# window, so the out-of-sample period is 1973-07 to 2004-11.
DGU_EXPECTED_T = 497           # months from 1963-07 to 2004-11 inclusive; a count, checked in notebook 01
DGU_EXPECTED_OOS = 377         # DGU_EXPECTED_T - DGU_ESTIMATION_WINDOW

# The replication extended to the latest month (notebook 07), fixed 1 October
# 2026 before the notebook's code: the nine rules on every month the vintage
# carries, reported for the paper's out-of-sample period, the months since the
# paper and the whole period. The expectations are in the notebook's introduction.
EXTENDED_NEW_PERIOD_START = "2004-12"   # the month after DGU's sample ends

# ---------------------------------------------------------------------------
# 4. DGU's analytical result and simulation (notebook 04)
# ---------------------------------------------------------------------------

# The analytical critical window (DGU Proposition 1, pp. 1937-1938; Figure 1, p. 1940).
# Panel E is the one calibrated to US stock-market data and the one the abstract quotes.
ANALYTIC_SHARPE_TANGENCY = 0.15    # S*, DGU Figure 1 panel E
ANALYTIC_SHARPE_1N = 0.12          # S_ew, DGU Figure 1 panel E
EXPECT_CRITICAL_M_LOWER = {25: 3000, 50: 6000}   # DGU p. 1941: "more than 3000 months" and "more than 6000 months"; the check is order of magnitude and direction in N

# The simulated dataset (DGU section 5.1, pp. 1941-1942).
SIM_RF_MEAN_ANNUAL = 0.02
SIM_RF_SD_ANNUAL = 0.02
SIM_FACTOR_MEAN_ANNUAL = 0.08
SIM_FACTOR_SD_ANNUAL = 0.16
SIM_ALPHA = 0.0
SIM_BETA_RANGE = (0.5, 1.5)          # loadings evenly spread
SIM_IDIO_VOL_RANGE = (0.10, 0.30)    # annual idiosyncratic volatility, uniform, cross-sectional mean 0.20
SIM_N = (10, 25, 50)
SIM_M = (120, 360, 6000)
SIM_T = 24000                        # months in one simulated history
SIM_SEED = 20260910                  # fixed so the run is reproducible; DGU's seed is unknown
# In one 24,000-month history the standard error of a monthly Sharpe ratio is
# about 1/sqrt(24000) = 0.0065, so the comparison with DGU Table 6 uses the same
# band as the empirical tables, plus the sign of (mv minus 1/N) in every cell.
BAND_SIM_SHARPE_ABS = BAND_SHARPE_ABS
# Table 6 reports 1/N and "mv (true)" as full-sample Sharpe ratios and the
# estimated rules out of sample; the comparison follows that convention. As a
# second check every rule is rerun on alternative draws of the market (seeds 1
# to SIM_SPREAD_DRAWS) and the published value is checked against their range.
SIM_SPREAD_DRAWS = 8

# Fat tails (notebook 05), an addition to the paper, fixed 30 September 2026
# before the notebook's code. The market above is rerun with every shock drawn
# from a Student-t distribution, scaled so that the mean and covariance of
# returns are exactly those of the normal market; one scale draw per month is
# shared by all shocks (a multivariate t), so an extreme month hits all assets
# at once. The normal shocks are the same draws as notebook 04's.
SIM_T_DOF = (10, 5)                  # degrees of freedom. 10: excess kurtosis 1, about a broad index's monthly returns. 5: excess kurtosis 6, about an industry portfolio's. At 4 or below the fourth moment is infinite and the standard error of a Sharpe ratio is undefined.
SIM_T_COMMON_SCALE = True            # one scale per month shared by all shocks (multivariate t); False draws one scale per shock series, which changes the shape of the month's co-movement as well as its size
# The crossing table: sample-based mean-variance against 1/N over a grid of
# windows, to locate the window at which the gap closes under each distribution.
SIM_CROSSING_WINDOWS = (120, 240, 360, 600, 1000, 2000, 3000, 4000, 6000, 9000, 12000)
FAT_TAIL_MOMENT_CHECK_ABS = 0.013    # notebook 05, expectation 1: two standard errors of a 24,000-month Sharpe ratio
FAT_TAIL_DIRECTION_SLACK = 0.01      # notebook 05, expectation 2: a cell counts as in the expected direction if Sharpe(t) <= Sharpe(normal) + this
# One test added after notebook 05's first run (6 October 2026), fixed here before
# its code and labelled in the notebook: the fat-tailed market rerun with one
# scale per shock series (common_scale False), so that an extreme month changes
# the shape of the estimated covariance as well as its level. Expectation P1a:
# the mean change in Sharpe ratio across the seven estimated rules and the nine
# cells of the Table 6 grid is more negative than under the common scale at both
# degrees of freedom. P1b: the rules whose weights depend on the shape of the
# covariance and use no means lose more on average than the other estimated rules.
FAT_TAIL_SHAPE_RULES = ("min", "min-c", "g-min-c")   # P1b: the minimum-variance rules

# Volatility that changes through time (notebook 06), the second addition to
# the paper, fixed 1 October 2026 before the notebook's code. The market of
# notebook 04 is rerun with shocks whose volatility follows a GARCH(1,1) process
# (Bollerslev 1986); the factor shock and every asset's idiosyncratic shock get
# their own process with the same parameters, so both the level and the shape
# of the covariance matrix move through time, while the unconditional mean and
# covariance are exactly those of the normal market (omega = 1 - alpha - beta).
# The expectations of notebooks 05 and 06 are in their introductions and use
# the two levels above.
SIM_GARCH_ALPHA = 0.10               # weight of last month's squared shock; a textbook monthly value
SIM_GARCH_BETA = 0.85                # weight of last month's variance; alpha + beta = 0.95 is the persistence, a half-life of a volatility shock of about 13 months
SIM_GARCH_OWN_PROCESS_PER_ASSET = True   # the factor and each idiosyncratic shock follow their own process; False would scale all shocks by one common volatility, which leaves the shape of the covariance unchanged

# ---------------------------------------------------------------------------
# 5. The extension: constrained index tracking on the 49 industries
# ---------------------------------------------------------------------------

EXT_UNIVERSE = "ind49"
EXT_ESTIMATION_WINDOW = 120        # the same window as the replication, so one convention runs through the repository
EXT_REBALANCE = "monthly"
# The extension sample starts at the first month in which all 49 industries carry
# a return (French codes the early gaps as -99.99): Hlth is the last industry to
# fill and does so in 1969-07. Notebook 01 asserts it on every run.
EXT_SAMPLE_START = "1969-07"
EXT_SAMPLE_END = None              # the latest month the vintage carries

# The benchmark (the index the portfolio tracks): the cap-weighted combination
# of the 49 industries, weights from firm count times average firm size, must
# track the market factor plus the risk-free rate within this tracking error a
# year over the full sample (notebook 08). If it does not, the derived weights
# are dropped and the benchmark is the market series.
CAPW_TE_MAX_ANNUAL = 0.0050

# Notebook 08, the cap-weight check, fixed 1 October 2026 before its code. The
# weight of industry i in month t is its firm count times its average firm size
# in French's file for month t, divided by the sum over the 49 industries;
# French measures both at the start of month t, so the weight uses nothing
# unknown when the month begins. The previous-month reading is run beside it as
# a check of this timing, and the notebook stops if it tracks more closely.
CAPW_WEIGHT_TIMING = "same_month"
CAPW_WEIGHT_TIMING_ALTERNATIVE = "previous_month"   # reported beside the chosen timing, never used as the benchmark
CAPW_MARKET_SERIES = ("Mkt-RF", "RF")                # the market series is Mkt-RF plus RF, so that both sides are plain returns
CAPW_REPORT_MONTHS = ("first", "2004-11", "last")

# The direct test of where the benchmark's gap to the market comes from
# (notebook 08, second part), fixed 1 October 2026 before its code. French forms
# the industry portfolios once a year at the end of June from the firms that
# have an SIC code then, while the market return takes in a firm from its first
# full month of trading, so between one July and the next the market holds
# firms that no industry portfolio holds. The test needs the firm list of each
# portfolio each month, which French does not publish, so it runs on the CRSP
# monthly stock file (WRDS, under Queen Mary University of London's
# subscription), the one use of data outside French's library in the study.
# The CRSP data stay on the machine that holds them: no notebook reads them,
# the repository carries none, and only the aggregates in
# DIRECT_TEST_RESULTS_FILE are published, with CRSP cited by name. The market
# universe is CRSP share codes 10 and 11 on NYSE, AMEX and NASDAQ, weighted by
# start-of-month market equity; the June-formed universe is the firms of that
# market that were in it at the end of June of the formation year with a
# positive SIC code. The script is checks/crsp_direct_test.py; its pure
# functions are in bp.direct_test.
DIRECT_TEST_SHARE_CODES = (10, 11)
DIRECT_TEST_EXCHANGE_CODES = (1, 2, 3)
DIRECT_TEST_FORMATION_MONTH = 6          # the end of June
DIRECT_TEST_MARKET_TE_MAX = 0.0020       # expectation 1: the CRSP market universe tracks French's market return within 20 basis points a year
DIRECT_TEST_MIN_CORRELATION = 0.80       # expectation 2: correlation of the predicted gap with the observed one
DIRECT_TEST_SLOPE_RANGE = (0.8, 1.2)     # expectation 2: slope of the observed gap on the predicted one
DIRECT_TEST_RESIDUAL_TE_MAX = 0.0025     # expectation 4: tracking error of what the explanation leaves, 25 basis points a year
DIRECT_TEST_RESULTS_FILE = "checks/results_crsp_direct_test.json"   # aggregates only; committed

# Constraints. Fixed 10 September 2026; the sources for the levels are in
# notebook 11, "Where the numbers come from".
ACTIVE_WEIGHT_BOUND = 0.02         # each untilted industry within 2 percentage points of its benchmark weight; an absolute cap does not fit 49 industries whose benchmark weights run from under 0.5% to over 10%
TURNOVER_CAP_MONTHLY_ONE_WAY = 0.02   # one-way turnover per rebalance, against drifted weights; 2% a month is about 24% a year, the enhanced-indexing range
TILT_INDUSTRIES = ("Coal", "Oil", "Util")   # French's column names, stripped of padding
TILT_MAX_SHARE_OF_BENCHMARK = 0.5  # each tilted industry held at no more than half its benchmark weight, a 50% reduction relative to the benchmark; a tilted industry's bounds are [0, 0.5 * benchmark], and ACTIVE_WEIGHT_BOUND does not apply to it, because the two would conflict for any industry above 4% of the benchmark
LONG_ONLY = True

# Constraint sets, cumulative, reported in this order. C3 is the mandate: the
# tilt with beta neutrality and the exclusions. Beta neutrality and the
# exclusions were added on 4 October 2026, after notebook 11's checks D1 to D4
# and before notebook 12's code, because the tilt's hedge left a beta bet
# standing when the covariance was noisy and bought tobacco and weapons. The
# tilt alone is CONSTRAINT_SET_TILT_ONLY, the C3 of the first design, reported
# beside the mandate in notebooks 11 and 12 so that the cost of the two
# additions is visible.
CONSTRAINT_SETS = {
    "C0": ("long_only",),
    "C1": ("long_only", "active_weight_bound"),
    "C2": ("long_only", "active_weight_bound", "turnover_cap"),
    "C3": ("long_only", "active_weight_bound", "turnover_cap", "tilt", "beta_neutral", "exclusion"),
}
CONSTRAINT_SET_TILT_ONLY = ("long_only", "active_weight_bound", "turnover_cap", "tilt")   # the C3 of the design of 10 September 2026
MANDATE_EXCLUSIONS = ("Smoke", "Guns")   # tobacco and weapons, the standard exclusions of Dutch institutional mandates, in French's industry names; weight zero

# The data the covariance estimators see, fixed 1 October 2026. The 120-month
# window is the paper's convention and gives 49 x 120 = 5,880 returns for the
# 1,225 entries of a 49-industry covariance matrix, fewer than five per entry.
# Practitioners estimate from daily returns (three years hold about 756 trading
# days, 30 observations per entry), impose structure, and weight recent days
# more; the extension does all three. Each estimator is built at two data
# frequencies: "monthly_120", the last EXT_ESTIMATION_WINDOW months of monthly
# excess returns, and "daily_3y", the last EXT_COV_DAILY_WINDOW_YEARS years of
# daily excess returns scaled to a monthly covariance by DAILY_TO_MONTHLY_SCALE
# (notebook 10 tests that scaling). A trading day on which any industry has no
# return (one in the sample, Softw on 1971-03-11) is left out of the window.
EXT_COV_FREQUENCIES = ("monthly_120", "daily_3y")
EXT_COV_DAILY_WINDOW_YEARS = 3
EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY = (1, 5)   # reported for the sample estimator only, no verdict
DAILY_TO_MONTHLY_SCALE = 21                        # 252 trading days a year divided by 12
EXT_COV_EWMA_HALFLIFE_DAYS = 252                   # the recency-weighted sensitivity: a half-life of one year of trading days, the RiskMetrics setting of Dom, Howard, Jansen and Lohre (2024, section 3.3.2); 126 until 3 October 2026, changed on reading that paper, before notebook 10's code

# Covariance estimators. The shrinkage target follows Dom, Howard, Jansen and
# Lohre (2024), the paper the design named as its reference, checked on
# 3 October 2026 before notebook 10's code: their linear shrinkage pulls the
# sample covariance towards a scaled identity matrix (Ledoit and Wolf 2004b),
# and their footnote 11 reports that the constant-correlation target gave
# inferior results. The same paper estimates from three years of daily returns
# and rebalances monthly, and finds that under long-only constraints the
# differences between estimators shrink, which is notebook 10's expectation 3.
COV_ESTIMATORS = ("sample", "ledoit_wolf", "factor", "pca")
LW_TARGET = "scaled_identity"          # Ledoit and Wolf (2004b), "A well-conditioned estimator for large-dimensional covariance matrices"; was "constant_correlation" until 3 October 2026
LW_TARGET_SENSITIVITY = "constant_correlation"   # Ledoit and Wolf (2004a), "Honey, I Shrunk the Sample Covariance Matrix"; reported, no verdict
FACTOR_MODEL_FACTORS = ("Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom")   # the Fama-French five factors plus momentum; the estimator is B F B' plus a diagonal of residual variances, B estimated in the same estimation window
PCA_N_COMPONENTS = 5                   # five components plus a diagonal idiosyncratic term
PCA_PC1_MARKET_CORR_MIN = 0.95         # notebook 09, expectation 2: the first component's correlation with the cap-weighted market return
PCA_USE_COVARIANCE = True              # covariance, not correlation, of monthly excess returns
PCA_SIGN_RULE = "positive market loading"
PCA_WINDOWS = {                        # three pre-specified windows; loading stability is the finding and carries no pass or fail
    "to_2019":   (None, "2019-12"),
    "2020_2022": ("2020-01", "2022-12"),
    "from_2023": ("2023-01", None),
}

# Notebook 09, the principal components of the 49 industries, fixed 1 October
# 2026 before its code; the expectations are in its introduction.
PCA_REPORT_COMPONENTS = 10
PCA_PARALLEL_DRAWS = 200
PCA_PARALLEL_PERCENTILE = 95
PCA_PARALLEL_SEED = 20260910
# Set after notebook 09 ran, as its design said: the first component's
# correlation with the benchmark fell below PCA_PC1_MARKET_CORR_MIN in the
# window from 2023 (0.837), so notebook 10's PCA estimator takes the
# benchmark's excess return as its first factor. Parallel analysis found two
# components above noise; PCA_N_COMPONENTS stays at five and the two-component
# version is reported beside it.
PCA_FIRST_COMPONENT = "market"        # notebook 10: the benchmark's excess return is the first factor; "data" would take it from the eigenvectors
PCA_N_COMPONENTS_SENSITIVITY = 2      # the parallel-analysis count on the full sample, reported beside PCA_N_COMPONENTS

# Notebook 10, the four covariance estimators, fixed 3 October 2026 before its
# code. The estimators, data frequencies, tests and expectations are stated in
# the notebook's introduction and in covariance.py.
COV_EVAL_START = "1979-07"            # 1969-07 plus 120 months; the three-year daily window starts 1972-07 and is the shorter requirement
COV_TEST_ACTIVE_DRAWS = 200           # random active-weight vectors for the calibration test
COV_TEST_SEED = 20260910
COV_LONG_ONLY_RANGE_MAX = 0.01        # expectation 3: one percentage point of annualised volatility across the four estimators on daily data
COV_SCALE_RATIO_RANGE = (0.8, 1.25)   # expectation 4: the ratio of the benchmark's realised monthly variance to 21 times its daily variance
# Three tests added after notebook 10's first run (4 October 2026), each fixed
# here before its code and labelled in the notebook: P1, a scale correction by
# the variance ratio of Lo and MacKinlay (1988), one number from the benchmark's
# daily returns that multiplies the daily covariance in place of
# DAILY_TO_MONTHLY_SCALE alone; P2, bias statistics by decade; P3, trading costs
# of the minimum-variance portfolios at EXT_COST_LEVELS.
COV_SCALE_VR_HORIZON_DAYS = DAILY_TO_MONTHLY_SCALE   # the horizon of the variance ratio: one month of trading days
COV_SCALE_VR_WINDOW_YEARS_SENSITIVITY = 10           # P1 sensitivity: the variance ratio from ten years of daily benchmark returns
COV_POST_RUN_DECADES_MIN_CLOSER = 4                  # P1a: of five decades, the corrected forecast is closer to calibrated in at least this many
COV_POST_RUN_COST_MIN_VARIANTS = 6                   # P3a and P3b: of eight main variants, the long-only portfolio beats the unconstrained one net of cost in at least this many

# Notebook 11, the optimiser, fixed 4 October 2026 before its code; the
# constraints themselves date from 10 September. The problem, the infeasible-month
# rule and the expectations are stated in the notebook's introduction and in
# optimiser.py.
OPT_SOLVER = "CLARABEL"                  # interior-point solver shipped with cvxpy; deterministic
OPT_SOLVER_OPTIONS = {"tol_gap_abs": 1e-12, "tol_gap_rel": 1e-12, "tol_feas": 1e-10}   # tighter than the solver's defaults (1e-8), which left weights 1e-4 off an independent solver's on a test problem
OPT_SOLVER_OPTIONS_ROBUST = {"tol_gap_abs": 1e-9, "tol_gap_rel": 1e-9, "tol_feas": 1e-9}   # the robust problem is written with second-order cones, on which the solver reaches 1e-9 and reports 1e-12 as inaccurate; the objective agrees with the 1e-12 run to 1e-10 on the test problems
OPT_TOL = 1e-6                           # feasibility checks on weights, bounds and turnover; also "binding" means within this of the limit
OPT_TOL_TE = 1e-6                        # expectation 2a: a forecast tracking error below this, per year, counts as zero
OPT_TILT_TE_MAX_ANNUAL = 0.01            # expectation 3: 100 basis points a year, the enhanced-indexing ex-ante limit (Robeco's composite ran at 1.06%, J.P. Morgan quotes below 2%; sources in notebook 11)
OPT_REPORT_MONTHS = ("first_eval", "2004-11", "last")   # first_eval is COV_EVAL_START, the first month with both windows
OPT_SPEED_LIMIT_STEPS = (1, 6, 12, 24)   # expectation 5: rebalances reported on the journey from 1/N to the benchmark
# Checks D1 to D4 on the practical problem behind the tilt's hedge, added after
# notebook 11's first run (4 October 2026), each fixed here before its code.
OPT_D1_MARKET_SHARE_MAX = 0.05         # D1a: the market part of the tilt's forecast active variance is below this share in every month-estimator pair
OPT_D2_TE_RISE_MAX = 0.0001            # D2a: a beta-neutral constraint raises the forecast tracking error by less than one basis point a year in every pair
OPT_D3_EXCLUDE = ("Smoke",)            # D3: the hedge's tobacco purchase refused, its cost reported
# The fixes F1 to F3 (4 October 2026, before notebook 12's code): beta neutrality
# and the exclusions join C3 (above), and a robust portfolio, the weights whose
# largest forecast active variance across the estimators' covariances is
# smallest (Goldfarb and Iyengar 2003; Tutuncu and Koenig 2004), joins the
# rolling evaluation as a variant. Its beta neutrality holds under the average
# of the five beta vectors, because one equality per covariance was infeasible
# at 1979-07 under the mandate's bounds.
OPT_FIX_RISE_MAX = 0.0015              # F2: the mandate's C3 costs less than 15 basis points a year more than the tilt alone
ROBUST_VARIANT = ("robust", "all_five")   # the robust portfolio: minimax over the five covariances
# Check D5 on the solver, added after notebook 12's first run (6 October 2026),
# fixed here before its code: at this month the index moved more than the cap
# allows, no portfolio satisfied the cap and the tilt at once, and the solver
# reported "optimal" with weights that fail the independent check. The
# optimiser therefore finds the smallest feasible turnover first and never
# poses a capped problem that has no solution; the check of every returned
# portfolio stays.
OPT_D5_MONTH = "1984-07"

# Notebook 12, the rolling evaluation, fixed 4 October 2026 before its code: one
# path per specification, each portfolio built on the data before its month,
# judged on realised tracking error, calibration of the forecasts, active
# return net of costs, turnover and holdings. The expectations are in the
# notebook's introduction.
EVAL_TE_RANGE_MAX = 0.0010           # expectation 2: 10 basis points a year across the four daily estimators
EVAL_BIAS_RANGE = (0.8, 1.25)        # expectation 3, the same band as COV_SCALE_RATIO_RANGE
EVAL_RELAX_SHARE_MAX = 0.10          # expectation 6: at most one month in ten with a relaxed cap
EVAL_START_PORTFOLIO = "benchmark"   # every path starts as the index
EVAL_BENCHMARK_TOL = 1e-6            # expectation 1: an active return below this a month counts as holding the benchmark; written as 1e-10 and set to the solver's precision (about 1e-7) before the recorded run
# Expectations 8 and 9, added after notebook 12's first run (6 October 2026),
# each fixed here before its code and labelled in the notebook. RiskMetrics
# joins as a seventh variant because Dom, Howard, Jansen and Lohre (2024) find
# that time dynamics matter more than shrinkage or factor structure; the
# calibration multiplies each forecast by the ratio of realised to forecast
# tracking error over the previous EVAL_CALIBRATION_WINDOW months of the same
# path, using only months before the forecast.
EWMA_VARIANT = ("sample", "daily_5y_ewma")   # the seventh variant: RiskMetrics, as in notebook 10's sensitivity
EVAL_CALIBRATION_WINDOW = 60                 # months of a path's own past the calibration uses; a standard deviation from 60 observations has a standard error of about 9%
EVAL_CALIBRATION_MIN = 24                    # the forecast is left as it is while fewer months are available
EVAL_CALIBRATED_BIAS_RANGE = (0.85, 1.15)    # expectation 9: the calibrated bias statistic under C3, four daily estimators

# Notebook 13, the factor attribution, fixed 6 October 2026 before its code: each
# path's monthly active return split into the contributions of the six factors
# (the active exposure to each, from the industries' betas estimated on the data
# before the month, times the factor's return in the month) plus a residual, the
# industries' own returns; the identity is the check. The paths attributed are
# the seven variants under the mandate's C3 and under the tilt alone, rebuilt
# from the modules; the expectations are in the notebook's introduction.
ATTRIBUTION_FACTORS = FACTOR_MODEL_FACTORS
ATTRIBUTION_IDENTITY_MAX_ABS_ERROR = 1e-9        # expectation 1: contributions plus residual minus active return, per month, in return units
ATTRIBUTION_SETS = ("C3", "C3 tilt only")        # the two constraint sets whose paths are attributed
ATTRIBUTION_REPRODUCTION_TOL = 0.00001           # expectation 1: the rebuilt paths reproduce notebook 12's realised tracking errors within 0.1 basis point a year
ATTRIBUTION_REFERENCE_TE = {                     # notebook 12's realised tracking errors, per year, for the reproduction check
    ("sample, daily 3y", "C3"): 0.0084654, ("ledoit_wolf, daily 3y", "C3"): 0.0084820, ("factor, daily 3y", "C3"): 0.0093449,
    ("pca, daily 3y", "C3"): 0.0091482, ("sample, monthly 120", "C3"): 0.0095761, ("robust, all five", "C3"): 0.0087358,
    ("RiskMetrics, daily 5y", "C3"): 0.0084278,
    ("sample, daily 3y", "C3 tilt only"): 0.0080276, ("ledoit_wolf, daily 3y", "C3 tilt only"): 0.0080282, ("factor, daily 3y", "C3 tilt only"): 0.0088296,
    ("pca, daily 3y", "C3 tilt only"): 0.0086722, ("sample, monthly 120", "C3 tilt only"): 0.0090924, ("robust, all five", "C3 tilt only"): 0.0082305,
    ("RiskMetrics, daily 5y", "C3 tilt only"): 0.0079698,
}
ATTRIBUTION_MARKET_EXPOSURE_MAX = 0.05           # expectation 2: the mean active exposure to Mkt-RF, in absolute value, for every path
ATTRIBUTION_SIGN_AGREEMENT_MIN = 4               # expectation 3: of the five factors other than the market, the number whose mean active exposure under the mandate has the same sign for all seven variants
ATTRIBUTION_FACTOR_SHARE_MAX = 0.5               # expectation 4: the share of the active return's variance the six factors explain, for every path
ATTRIBUTION_FACTOR_PART_MAX = 0.5                # expectation 5a: the share of the mandate's cost against the tilt alone that comes through the factor contributions, every variant
ATTRIBUTION_ARITHMETIC_TOL = 0.0002              # expectation 5b: the exclusions' factor part (the five factors other than the market) within this of the arithmetic of the inputs, per year (2 basis points), every variant

# Specification count, written down before the code runs and reported.
EXT_SPEC_COUNT = (len(COV_ESTIMATORS) + 3) * len(CONSTRAINT_SETS)   # 28 optimised specifications: four daily-based estimators, the monthly sample covariance, the robust portfolio and RiskMetrics, times four constraint sets
EXT_COST_LEVELS = (DGU_TCOST, TCOST_SENSITIVITY)              # each reported net of cost at both levels; the cost is applied after the fact and does not enter the optimisation

# ---------------------------------------------------------------------------
# 6. Output conventions
# ---------------------------------------------------------------------------

OUTPUT_DIR = "outputs"
PROVENANCE_FILE = "outputs/provenance_french.json"   # download date, URL, SHA-256 and CRSP vintage line per file
TIMEZONE = "Europe/Amsterdam"

# ---------------------------------------------------------------------------
# 7. Version 2, module A: the factor objective under the mandate
# ---------------------------------------------------------------------------
# Fixed 8 October 2026, before any of the module's code was written, from the
# design of 7 October 2026 and version 1's results at commit 6835d49. Version 1 found that
# the mandate's tilt leaves incidental exposures of 0.004 to 0.018 in absolute
# value to the six factors, costing 9 to 13 basis points a year. Module A makes
# the exposures the objective: each month, with the data available before the
# month, the optimiser maximises the sum of the active exposures to the four
# targeted factors, s' B' (w - b), where B holds the industries' betas to the
# six factors estimated as in notebook 13 (three years of daily returns before
# the month) and s is FO_TARGET_SIGN, under every constraint of the mandate
# (CONSTRAINT_SETS["C3"]) and one new constraint, the budget: the forecast
# tracking error from the month's covariance estimate at most FO_BUDGETS_ANNUAL.
# The study holds no view on expected returns, as version 1 did not: the
# targets are the four of French's six factors with a documented premium, the
# direction of each is the literature's, and the four are weighted equally in
# beta units. Nothing estimates a premium or times a factor.
# Naming, fixed in the design: "tilt" stays the industry reduction of version 1;
# the new objective is the factor objective, the four are the targeted factors,
# the limit is the budget.
FO_DECISION_DATE = "2026-10-08"
FO_TARGET_FACTORS = ("HML", "RMW", "CMA", "Mom")              # value, profitability, investment, momentum
FO_TARGET_SIGN = {"Mkt-RF": 0.0, "SMB": 0.0, "HML": 1.0, "RMW": 1.0, "CMA": 1.0, "Mom": 1.0}   # s, on ATTRIBUTION_FACTORS; +1 is the direction of the documented premium
FO_CONSTRAINT_SET = "C3"                                       # the mandate: every constraint of version 1's C3, unchanged
FO_BUDGETS_ANNUAL = (0.0075, 0.0100, 0.0150)                   # tau, per year: just above the mandate's own realised cost (84.7 to 95.8 bp), the enhanced-indexing limit of OPT_TILT_TE_MAX_ANNUAL, and the generous case
FO_BUDGET_MAIN = 0.0100                                        # the budget at which the robust, RiskMetrics and single-factor paths run and most expectations are tested
FO_BETA_WINDOW_YEARS = EXT_COV_DAILY_WINDOW_YEARS              # the betas' window: notebook 13's three years of daily returns before the month
# Infeasibility rule: in a month in which the budget cannot be met under the
# mandate (the tracking-error-minimising solution of version 1 already exceeds
# tau), the path takes that minimising solution, the month is counted and
# reported, and nothing else is relaxed; the count is one of the results.
FO_INFEASIBLE_RULE = "tracking-error minimiser"
# The 19 paths, the count fixed here: the five estimator variants of version 1
# at each of the three budgets (15); the robust and RiskMetrics variants at the
# main budget (2); two single-factor sensitivities at the main budget with the
# daily sample estimator, s on value alone and on momentum alone (2). Version
# 1's mandate paths (tracking error minimised, no objective) are the comparison
# and are not rerun. A path added after the first run is disclosed with its
# date and reason, and the count restated.
FO_SINGLE_FACTOR_PATHS = (("HML",), ("Mom",))
FO_PATH_COUNT = 19
# The two measures new to module A. The exposure the budget bought: the mean
# over the months of the sum of the four targeted active exposures, divided by
# the realised tracking error in basis points a year. The gap: for each month,
# the targeted exposure the optimiser chose ex ante, s' B_t' a_t with B_t the
# betas estimated before the month, minus the exposure the same weights carry
# when the betas are estimated after the month, s' B_{t+1}' a_t; its mean
# absolute value over the months is the path's gap.
FO_GAP_DEFINITION = "s'(B_t - B_{t+1})' a_t, mean absolute value over the months"
# The seven expectations, each a strict pass or fail; the levels from
# version 1's results at 6835d49, with the design's reasons.
# E1 and E2 were restated on 9 October 2026, after notebook 14's check of
# the arithmetic and before any path ran, by the design of 9 October 2026:
# a level changed at that point, between the pre-run check notebook 14
# exists for and the first rolling run, is a pre-run restatement, dated and
# disclosed here and once in the write-up; a level changed after notebook 15
# runs is a post-run change. The original levels of 8 October stay below,
# marked as superseded.
FO_RESTATEMENT_DATE = "2026-10-09"
# E1. At the main budget the budget binds (forecast tracking error within
# FO_E1_BIND_TOL of the budget) in at least FO_E1_BIND_SHARE_MIN of the
# months in which the mandate's own minimum forecast tracking error, with
# turnover free, is at or below the budget, for each of the four daily
# estimators. The months the mandate alone cannot meet are reported per
# estimator and budget from notebook 14's feasibility table; the months the
# path misses beyond them, with the turnover cap in force, are counted and
# reported, since that difference is what the path adds. Reason for the
# restatement: notebook 14 found the mandate alone above 100 bp in 37
# (sample), 43 (Ledoit-Wolf) and 39 (six-factor) of the 566 months, 22 for
# principal components, nearly all in 1979 to 1989, so the budget cannot
# bind in those months whatever the objective.
FO_E1_BIND_TOL_ANNUAL = 0.0001           # one basis point a year
FO_E1_BIND_SHARE_MIN = 0.95
FO_E1_DENOMINATOR = "months the mandate alone can meet"
# Superseded on 9 October 2026: the same share of all 566 months. Reason
# given on 8 October: version 1's forecast under the mandate was 55 to 68 bp
# on average and 47 to 55 for the tilt alone at the 1979 oil peak for the
# daily estimators, so a budget of 100 should be reachable almost always;
# the monthly estimator (108 at that peak) may fail it in the first years.
FO_E1_DENOMINATOR_SUPERSEDED_2026_10_08 = "all 566 months"
# E2. At the main budget the mean active exposure to each targeted factor is
# positive on every main path, the average of the four is at least
# FO_E2_MEAN_EXPOSURE_MIN for the daily sample estimator, and that average
# rises with the budget (75, then 100, then 150) on every estimator; the four
# exposures are reported per path at every budget, the average never alone.
# Reason for the restatement: notebook 14's stationary mean of the four at
# 100 bp is 0.030 for the daily sample estimator and a capped path holds at
# most the stationary exposure on average; the level is four fifths of
# 0.030, the fifth being the design's allowance for what the turnover cap,
# the infeasible months and the rolling betas may cost against the
# stationary solution, a judgement stated as such. Under an active weight
# bound of 2 percentage points on 49 industries, 100 bp of budget buys about
# 0.03 of average exposure in beta units.
FO_E2_MEAN_EXPOSURE_MIN = 0.024
# Superseded on 9 October 2026. Reason given on 8 October: version 1's
# incidental exposures were 0.004 to 0.018 in absolute value with no
# objective; 49 industries within 2 percentage points of their weights can
# reach three to ten times that when the objective asks for it.
FO_E2_MEAN_EXPOSURE_MIN_SUPERSEDED_2026_10_08 = 0.05
# One diagnostic per path, counted and reported with no expectation
# attached (the design of 9 October 2026): the months in which any targeted
# exposure ends below its incidental value, the value the mandate's
# tracking-error minimiser carries. Notebook 14 found such a month in 5 of
# its 15 month-estimator pairs at 100 bp.
FO_DIAGNOSTIC_BELOW_INCIDENTAL = "months with a targeted exposure below its incidental value"
# E3. Realised tracking error exceeds the budget by 20 to 60 per cent on the
# daily estimators, and by more on the monthly sample covariance; tested at
# every budget (the design names none), the monthly ratio against the largest
# daily ratio at the same budget. Reason: version 1's bias was 1.22 to 1.39 on
# daily data and 1.61 on monthly; a budget-binding solution concentrates risk
# where the estimator sees least, so the miss may exceed version 1's, bounded
# here at 1.6.
FO_E3_REALISED_OVER_BUDGET_RANGE = (1.2, 1.6)
# E4. The four targeted factors' contributions sum to a positive amount on
# every main path, and at the main budget and above the six factors explain at
# least FO_E4_FACTOR_SHARE_MIN of the active return's variance on every main
# path. Reason: version 1's factors explained 14 to 28 per cent of incidental
# exposures; deliberate exposures raise the share.
FO_E4_FACTOR_SHARE_MIN = 0.30
# E5. At the main budget the gross active return is positive for at least
# FO_E5_POSITIVE_MIN of the five estimators; the net return at DGU_TCOST per
# unit of turnover lies within FO_E5_NET_GAP_MAX of the gross. Whether the
# active return is more than two standard errors from zero is reported, not
# expected. Reason: the study holds no view on the premia; the sign follows
# from the targets if the premia had their documented sign over the sample,
# which is what the data say and not a claim of skill.
FO_E5_POSITIVE_MIN = 4
FO_E5_NET_GAP_MAX = 0.0010               # 10 basis points a year
# E6. Of the two single-factor paths, momentum alone has the higher turnover
# and the larger gap, and the turnover cap binds in more months on the
# momentum path than on the value path. Reason: industry momentum betas rotate
# within a year; value betas do not.
# E7. At the main budget the spread across the four daily estimators in
# realised tracking error is at most FO_E7_TE_RANGE_MAX, the exposure bought
# per basis point differs across them by less than FO_E7_EXPOSURE_PER_BP_REL_MAX
# of their mean (largest minus smallest over the mean), and the monthly sample
# covariance remains the worst calibrated (the largest realised over forecast
# of the five). Reason: version 1, a range of 9 basis points on a level of
# about 90, the monthly covariance 61 per cent under-forecast.
FO_E7_TE_RANGE_MAX = 0.0020              # 20 basis points a year
FO_E7_EXPOSURE_PER_BP_REL_MAX = 0.30
# Notebook 14 runs the objective at the report months of version 1 and over
# the sample without a rolling path: a worked month, the exposure against the
# budget at FO_CURVE_BUDGETS (the three budgets and points between), and the
# feasibility of each budget in every month, from the forecast tracking error
# of the tracking-error-minimising solution under the mandate.
FO_CURVE_BUDGETS_ANNUAL = (0.0050, 0.0075, 0.0100, 0.0125, 0.0150, 0.0200)
FO_TOL_TE = 1e-9                         # a forecast tracking error within this of the budget counts as on it, in the tests

In [ ]:
%%writefile src/bp/french_loader.py
"""
Ken French data library: download, record, parse.

The library serves each dataset as a zip holding one CSV. The CSV is not a
single table. It is a text file with between two and ten blocks, each announced by a title
line ("  Average Value Weighted Returns -- Monthly"), followed by a header row
that starts with a comma, followed by rows that start with a date: six digits
for monthly data (192607), four for annual (1927). Missing values are coded
-99.99 or -999. Returns are in percent.

This module turns that into pandas DataFrames indexed by a monthly Period, in
decimal returns, with the missing codes replaced by NaN. It also writes a
provenance record for every file it downloads: the URL, the download time, the
SHA-256 of the zip, and the CRSP vintage line French prints at the top of each
CSV. A result is only as reproducible as the vintage it was computed on, and
French revises history, so the record is part of every output.

Nothing here is committed to the repository: the zips live in a local cache
directory that .gitignore excludes.
"""
from __future__ import annotations

import hashlib
import io
import json
import re
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import requests

from . import constants as C

_DATE_ROW = re.compile(r"^\s*(\d{8}|\d{6}|\d{4})\s*,")   # daily YYYYMMDD, monthly YYYYMM, annual YYYY


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def download(key: str, cache_dir: str | Path = "data/french", force: bool = False) -> dict:
    """Fetch one French zip by its key in constants.FRENCH_FILES.

    Returns the provenance record for the file and writes the zip to cache_dir.
    A cached zip is reused unless force=True, and its record is rebuilt from the
    cached bytes, so the SHA-256 always describes the file actually on disk.
    """
    filename = C.FRENCH_FILES[key]
    url = C.FRENCH_BASE_URL + filename
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / filename
    if force or not path.exists():
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
        fetched_now = True
    else:
        fetched_now = False
    data = path.read_bytes()
    now_utc = datetime.now(timezone.utc)
    record = {
        "key": key,
        "filename": filename,
        "url": url,
        "fetched_this_run": fetched_now,
        "download_time_utc": now_utc.isoformat(timespec="seconds"),
        "download_time_local": now_utc.astimezone(ZoneInfo(C.TIMEZONE)).isoformat(timespec="seconds"),
        "zip_sha256": sha256_bytes(data),
        "zip_bytes": len(data),
        "vintage_line": _vintage_line(data),
    }
    return record


def _csv_text(zip_bytes: bytes) -> str:
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if len(names) != 1:
            raise ValueError(f"expected one CSV in the zip, found {names}")
        return zf.read(names[0]).decode("latin-1")


def _vintage_line(zip_bytes: bytes) -> str:
    """The first line of the CSV, e.g. 'This file was created using the 202607 CRSP database.'"""
    text = _csv_text(zip_bytes)
    for line in text.splitlines():
        if line.strip():
            first = line.strip()
            return first.split(".")[0] + "." if "." in first else first
    return ""


def parse_blocks(text: str) -> list[dict]:
    """Split a French CSV into its blocks.

    Each block is returned as {"title": str, "frequency": "daily" | "monthly" |
    "annual", "frame": DataFrame} where the frame is indexed by Period (day,
    month or year) and holds floats in the file's own units (percent). The title is the last non-empty line before
    the header row that is not itself a data row.
    """
    lines = text.splitlines()
    blocks = []
    i = 0
    n = len(lines)
    while i < n:
        if _DATE_ROW.match(lines[i]):
            # Walk back to the header row (starts with a comma) and the title above it.
            header_idx = i - 1
            while header_idx >= 0 and not lines[header_idx].lstrip().startswith(","):
                header_idx -= 1
            if header_idx < 0:
                raise ValueError(f"data row at line {i} has no header row above it")
            title_idx = header_idx - 1
            while title_idx >= 0 and not lines[title_idx].strip():
                title_idx -= 1
            title = lines[title_idx].strip() if title_idx >= 0 else ""
            # The factor files and the later blocks of the size-and-value file
            # carry no title line, only prose; a sentence is not a title.
            if not title or title.endswith(".") or len(title) > 70:
                title = f"(untitled block {len(blocks)})"
            columns = [c.strip() for c in lines[header_idx].split(",")[1:]]
            rows = []
            j = i
            while j < n and _DATE_ROW.match(lines[j]):
                rows.append(lines[j])
                j += 1
            raw = pd.read_csv(io.StringIO("\n".join(rows)), header=None, index_col=0)
            raw.columns = columns[: raw.shape[1]]
            date_len = len(str(raw.index[0]).strip())
            if date_len == 8:
                idx = pd.PeriodIndex(pd.to_datetime(raw.index.astype(str), format="%Y%m%d"), freq="D")
                freq = "daily"
            elif date_len == 6:
                idx = pd.PeriodIndex([pd.Period(f"{str(d)[:4]}-{str(d)[4:6]}", freq="M") for d in raw.index])
                freq = "monthly"
            else:
                idx = pd.PeriodIndex([pd.Period(str(d), freq="Y") for d in raw.index])
                freq = "annual"
            frame = raw.astype(float)
            frame.index = idx
            frame.index.name = "period"
            blocks.append({"title": title, "frequency": freq, "frame": frame})
            i = j
        else:
            i += 1
    return blocks


def load_blocks(key: str, cache_dir: str | Path = "data/french") -> list[dict]:
    """All blocks of one file, with the file's provenance record attached to each."""
    record = download(key, cache_dir)
    text = _csv_text((Path(cache_dir) / C.FRENCH_FILES[key]).read_bytes())
    blocks = parse_blocks(text)
    for b in blocks:
        b["provenance"] = record
    return blocks


def load_monthly(key: str, block: int | str = 0, cache_dir: str | Path = "data/french",
                 as_decimal: bool = True) -> pd.DataFrame:
    """One block of one file, in decimal returns with missing codes as NaN.

    The name dates from the monthly files; the function serves the daily files
    in the same way (the block's index is then a daily Period).

    block is an integer position among the file's blocks, or a regular
    expression matched against block titles (the first match wins).
    """
    blocks = load_blocks(key, cache_dir)
    if isinstance(block, int):
        chosen = blocks[block]
    else:
        matches = [b for b in blocks if re.search(block, b["title"], flags=re.I)]
        if not matches:
            raise KeyError(f"no block in {key} matches {block!r}; titles: {[b['title'] for b in blocks]}")
        chosen = matches[0]
    frame = chosen["frame"].copy()
    for code in C.FRENCH_MISSING_CODES:
        frame = frame.mask(np.isclose(frame, code))
    if as_decimal and C.FRENCH_RETURNS_ARE_PERCENT:
        frame = frame / 100.0
    frame.attrs["title"] = chosen["title"]
    frame.attrs["provenance"] = chosen["provenance"]
    return frame


def load_counts(key: str, block: int | str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """A block in the file's own units (firm counts, average firm size), missing codes as NaN."""
    return load_monthly(key, block, cache_dir, as_decimal=False)


def write_provenance(records: list[dict], path: str | Path = C.PROVENANCE_FILE) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ordered = sorted(records, key=lambda r: r["key"])
    path.write_text(json.dumps(ordered, indent=2, sort_keys=True) + "\n")
    return path


def block_summary(key: str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """One row per block: title, frequency, columns, rows, first and last period."""
    rows = []
    for k, b in enumerate(load_blocks(key, cache_dir)):
        f = b["frame"]
        rows.append({
            "file": key, "block": k, "title": b["title"], "frequency": b["frequency"],
            "columns": f.shape[1], "rows": f.shape[0],
            "first": str(f.index[0]), "last": str(f.index[-1]),
        })
    return pd.DataFrame(rows)

In [ ]:
%%writefile src/bp/benchmark.py
"""The benchmark of the extension: the cap-weighted combination of French's 49 industry portfolios.

Each industry portfolio's value-weighted return for a month is the return of the
firms in the industry, each counted in proportion to its market equity at the
start of the month. Weighting the 49 industry returns by each industry's total
market equity at the same time (firm count times average firm size, two blocks of
the same French file) gives the value-weighted return of all the firms in the 49
portfolios, which should be close to French's market return. Notebook 08 measures
how close, against the level fixed in constants.CAPW_TE_MAX_ANNUAL, and decides
whether these weights are the benchmark of notebooks 09 to 13.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import constants as C
from . import french_loader as fl

PERIODS_PER_YEAR = 12


def load_inputs(cache_dir: str = "data/french") -> dict[str, pd.DataFrame | pd.Series]:
    """The three monthly blocks of the 49-industry file and the market series.

    returns: value-weighted industry returns (decimal); firms: number of firms;
    size: average firm size in millions of dollars; market: Mkt-RF plus RF from
    the factors file (decimal). Missing codes are NaN in all of them.
    """
    returns = fl.load_monthly("ind49", r"Value Weighted Returns -- Monthly", cache_dir)
    firms = fl.load_counts("ind49", r"Number of Firms", cache_dir)
    size = fl.load_counts("ind49", r"Average Firm Size", cache_dir)
    f3 = fl.load_monthly("factors3", 0, cache_dir)
    excess, rf = C.CAPW_MARKET_SERIES
    market = (f3[excess] + f3[rf]).rename("market")
    return {"returns": returns, "firms": firms, "size": size, "market": market}


def market_equity(firms: pd.DataFrame, size: pd.DataFrame) -> pd.DataFrame:
    """Total market equity of each industry, in the units of the size block (millions of dollars)."""
    if not firms.index.equals(size.index) or not firms.columns.equals(size.columns):
        raise ValueError("firm counts and average sizes must share months and industries")
    return firms * size


def cap_weights(firms: pd.DataFrame, size: pd.DataFrame, timing: str = C.CAPW_WEIGHT_TIMING) -> pd.DataFrame:
    """Each industry's share of total market equity, per month; the rows sum to one.

    timing "same_month": the count and size of month t weight the returns of
    month t (French measures both at the start of the month). "previous_month":
    the count and size of month t-1 weight the returns of month t; the first
    month has no weights. Any month in which an industry's count or size is
    missing has no weights at all, so that a partial sum never passes as a
    whole-market weight.
    """
    me = market_equity(firms, size)
    if timing == "previous_month":
        me = me.shift(1)
    elif timing != "same_month":
        raise ValueError(f"unknown timing {timing!r}")
    total = me.sum(axis=1, min_count=me.shape[1])
    return me.div(total, axis=0)


def combination_return(returns: pd.DataFrame, weights: pd.DataFrame) -> pd.Series:
    """Sum over industries of weight times return, per month; NaN where any of the 49 is missing."""
    if not returns.columns.equals(weights.columns):
        raise ValueError("returns and weights must have the same industries in the same order")
    aligned = weights.reindex(returns.index)
    return (aligned * returns).sum(axis=1, min_count=returns.shape[1]).rename("combination")


def tracking_error(a: pd.Series, b: pd.Series, periods_per_year: int = PERIODS_PER_YEAR) -> float:
    """Annualised standard deviation of the per-period difference a - b, over the periods both have."""
    d = (a - b).dropna()
    if len(d) < 2:
        return float("nan")
    return float(d.std(ddof=1) * np.sqrt(periods_per_year))


def compare(a: pd.Series, b: pd.Series, periods_per_year: int = PERIODS_PER_YEAR) -> dict[str, float]:
    """Tracking error, annualised mean difference, correlation, largest absolute monthly difference and month count of a against b."""
    both = pd.concat([a, b], axis=1).dropna()
    d = both.iloc[:, 0] - both.iloc[:, 1]
    return {
        "months": int(len(d)),
        "tracking_error_annual": float(d.std(ddof=1) * np.sqrt(periods_per_year)),
        "mean_difference_annual": float(d.mean() * periods_per_year),
        "correlation": float(both.iloc[:, 0].corr(both.iloc[:, 1])) if both.std(ddof=1).gt(0).all() else float("nan"),
        "max_abs_monthly_difference": float(d.abs().max()),
        "month_of_max": str(d.abs().idxmax()),
    }


def weight_extremes(weights: pd.DataFrame, month: str | pd.Period) -> dict[str, object]:
    """The largest and smallest weights in one month, with their industries."""
    row = weights.loc[pd.Period(month, "M")].dropna()
    return {
        "month": str(pd.Period(month, "M")),
        "largest": row.idxmax(), "largest_weight": float(row.max()),
        "smallest": row.idxmin(), "smallest_weight": float(row.min()),
        "industries": int(len(row)),
    }

## Count first: the three blocks of the 49-industry file

The file has eight blocks. Three are used here: the value-weighted monthly returns, the number of firms, and the average firm size. The cell loads them, restricts them to the extension sample, asserts expectation 1 and records the vintage.

In [ ]:
import sys
sys.path.insert(0, "src")
import numpy as np
import pandas as pd
from bp import constants as C
from bp import french_loader as fl
from bp import benchmark as BM

pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

print(fl.block_summary("ind49").to_string())
inputs = BM.load_inputs()
start = pd.Period(C.EXT_SAMPLE_START, "M")
returns = inputs["returns"].loc[start:]
firms = inputs["firms"].loc[start:]
size = inputs["size"].loc[start:]
market = inputs["market"].loc[start:]
LAST = returns.index[-1]

# Expectation 1: the same months in all three blocks, no missing value anywhere in the sample.
assert returns.index.equals(firms.index) and returns.index.equals(size.index), "the three blocks do not share the same months"
full = pd.period_range(start, LAST, freq="M")
assert len(returns) == len(full), f"{len(full) - len(returns)} months missing inside the sample"
for name, frame in (("returns", returns), ("firm counts", firms), ("average sizes", size)):
    assert frame.shape[1] == 49, f"{name}: {frame.shape[1]} industries"
    assert not frame.isna().any().any(), f"{name}: missing values inside the sample"
assert (firms > 0).all().all(), "an industry with no firms inside the sample"
assert market.loc[start:LAST].notna().all(), "the market series has a gap inside the sample"
records = [fl.download(key) for key in ("ind49", "factors3")]
fl.write_provenance(records)
print()
print(f"extension sample {start} to {LAST}: {len(returns)} months, 49 industries, 3 blocks, no missing value (expectation 1 holds)")
print(f"firms in the 49 portfolios: {int(firms.loc[start].sum()):,} in {start}, {int(firms.loc['2004-11'].sum()):,} in 2004-11, {int(firms.loc[LAST].sum()):,} in {LAST}")
total_me = (firms * size).sum(axis=1)
print(f"total market capitalisation of the 49 (millions of dollars): {total_me.loc[start]:,.0f} in {start}, {total_me.loc[LAST]:,.0f} in {LAST}")
print("vintage:", "; ".join(sorted({r["vintage_line"] for r in records})), "(provenance written to", C.PROVENANCE_FILE + ")")

## The weights and the combination, under both timings

`cap_weights` divides each industry's firm count times average size by the sum over the 49, per month. `combination_return` multiplies each month's weights by that month's industry returns and sums. The same-month timing is the design; the previous-month timing shifts the weights forward by one month and is run only to check the reading of French's timing.

In [ ]:
w_same = BM.cap_weights(firms, size, C.CAPW_WEIGHT_TIMING)
w_prev = BM.cap_weights(inputs["firms"], inputs["size"], C.CAPW_WEIGHT_TIMING_ALTERNATIVE).loc[start:]
# Under the previous-month timing the first month of the sample takes its weights from 1969-06, in which
# one industry has no firms yet, so that month has no weights; the timing check runs on the months both timings cover.
common = w_prev.dropna(how="all").index
assert np.allclose(w_same.sum(axis=1), 1.0) and np.allclose(w_prev.loc[common].sum(axis=1), 1.0)
combo_same = BM.combination_return(returns, w_same)
combo_prev = BM.combination_return(returns, w_prev)
assert combo_same.notna().all() and combo_prev.loc[common].notna().all()
print(f"same-month weights cover {combo_same.notna().sum()} months from {combo_same.index[0]}; previous-month weights cover {len(common)} months from {common[0]}")
side_by_side = pd.DataFrame({"market (Mkt-RF + RF)": market.loc[start:LAST], "same-month weights": combo_same, "previous-month weights": combo_prev})
print("first six months:")
print(side_by_side.head(6).to_string())
print()
print("last six months:")
print(side_by_side.tail(6).to_string())

## Expectations 2 and 3: the timing check and the gate

Tracking error is the standard deviation of the monthly difference, times the square root of 12. The level the gate uses, 50 basis points a year, corresponds to a typical monthly difference of 0.14%.

In [ ]:
cmp_same = BM.compare(combo_same, market)                        # the gate: every month of the sample
cmp_same_common = BM.compare(combo_same.loc[common], market)     # the timing check: the months both timings cover
cmp_prev = BM.compare(combo_prev.loc[common], market)
table = pd.DataFrame({"same-month weights, all months": cmp_same, "same-month weights, common months": cmp_same_common, "previous-month weights": cmp_prev})
print(table.to_string())
print()
te_same, te_prev = cmp_same_common["tracking_error_annual"], cmp_prev["tracking_error_annual"]
print(f"tracking error, same-month weights   : {te_same * 1e4:6.1f} basis points a year")
print(f"tracking error, previous-month weights: {te_prev * 1e4:6.1f} basis points a year")
assert te_same < te_prev, ("expectation 2 fails: the previous-month weights track the market more closely, so French's count and size "
                           "would describe the end of the month and the same-month weights would carry look-ahead; stop here")
print(f"expectation 2 holds: the same-month weights track the market more closely ({te_same * 1e4:.1f} against {te_prev * 1e4:.1f} basis points)")
te_gate = cmp_same["tracking_error_annual"]
gate = te_gate <= C.CAPW_TE_MAX_ANNUAL
print(f"expectation 3, the gate: {te_gate * 1e4:.1f} basis points over all {cmp_same['months']} months against the level of {C.CAPW_TE_MAX_ANNUAL * 1e4:.0f}: {'MET' if gate else 'NOT MET'}")
if gate:
    print("decision: the derived cap weights are the benchmark of notebooks 09 to 13, return series and industry weights alike")
else:
    print("decision: the benchmark return is the market series; the industry weights for the constraints go back to the design before notebook 09")

## Expectation 4: what the benchmark looks like

The largest and smallest industry weights at three dates, and the twelve months in which the combination and the market differ most.

In [ ]:
report_months = [start if m == "first" else (LAST if m == "last" else pd.Period(m, "M")) for m in C.CAPW_REPORT_MONTHS]
extremes = pd.DataFrame([BM.weight_extremes(w_same, m) for m in report_months]).set_index("month")
print(extremes.to_string())
print()
top = pd.DataFrame({m: w_same.loc[m].sort_values(ascending=False).head(8) for m in report_months})
top.columns = [str(c) for c in top.columns]
print("the eight largest industries at each date (weights):")
print(top.to_string())
print()
diff = (combo_same - market.loc[start:LAST]).rename("combination minus market")
worst = diff.abs().sort_values(ascending=False).head(12).index
print("the twelve months of largest absolute difference:")
print(pd.DataFrame({"market": market.loc[worst], "combination": combo_same.loc[worst], "difference": diff.loc[worst]}).sort_index().to_string())
print()
print(f"annualised mean difference: {cmp_same['mean_difference_annual'] * 1e4:+.1f} basis points a year; correlation {cmp_same['correlation']:.5f}")

## Where the difference comes from

This cell was written after the run, to describe the difference the gate measured; it changes no decision. French forms the industry portfolios once a year, at the end of June, from the firms that have an SIC code at that time. A firm that lists in, say, October joins the market return in November and joins an industry portfolio only the next July. If that is the source of the difference, the difference should be smallest in July and grow through the year, and it should be largest in the years with the most new listings. The cell prints the tracking error by calendar month and by decade.

In [ ]:
by_month = diff.groupby(diff.index.month).std(ddof=1) * np.sqrt(12) * 1e4
by_month.index = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
print("tracking error by calendar month, basis points a year (July is the first month after the portfolios are re-formed):")
print(by_month.reindex(["Jul", "Aug", "Sep", "Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun"]).round(0).astype(int).to_string())
print()
by_decade = diff.groupby((diff.index.year // 10) * 10).agg(["count", "std", "mean"])
by_decade["tracking error, bp a year"] = by_decade["std"] * np.sqrt(12) * 1e4
by_decade["mean difference, bp a year"] = by_decade["mean"] * 12 * 1e4
print("by decade:")
print(by_decade[["count", "tracking error, bp a year", "mean difference, bp a year"]].to_string(float_format=lambda v: f"{v:.1f}"))
print()
print(f"share of months in which the two differ by less than 0.14%: {(diff.abs() < 0.0014).mean():.1%}; by less than 0.30%: {(diff.abs() < 0.003).mean():.1%}")

## A direct test of the explanation, on firm-level data

The explanation above predicts two patterns and both appear, and a prediction that comes true is weaker evidence than a direct test: another cause could produce the same two patterns. The direct test needs the list of firms in each portfolio each month, which French does not publish, so it ran on the CRSP monthly stock file, the database French builds his files from, accessed through WRDS under Queen Mary University of London's subscription (Center for Research in Security Prices, LLC; download date in the results file). This is the one exception to this project's rule of using French's files only, decided for this test alone. The CRSP data stay on the machine that holds them: the script `checks/crsp_direct_test.py` reads them there, this notebook never reads them, and only the aggregate results are in the repository, in `checks/results_crsp_direct_test.json`, written into the working folder by the next cell so that this notebook can print them. The design and the four expectations were fixed in `constants.py` before the script was written (the `DIRECT_TEST_` constants).

The test. For every month, two sets of firms are built from CRSP. The market universe: every firm with CRSP share code 10 or 11 (ordinary common shares of US companies) on the NYSE, AMEX or NASDAQ (exchange codes 1, 2 and 3), with a price and shares outstanding at the end of the previous month and a return in the month. Weighted by that start-of-month market capitalisation, this is how French builds his market return. The June-formed universe: the firms of the market universe that were already in it at the end of the last June and had an SIC code then; this is who can be in an industry portfolio. The firms in the first set and not in the second are the ones the explanation blames: listed since June, or without an SIC code. The predicted gap is the value-weighted return of the market universe minus that of the June-formed universe. The observed gap is French's market return minus the 49-industry combination of this notebook. Expectation 1 checks that the CRSP market universe reproduces French's market return (tracking error at most 20 basis points a year; if not, the universe is built wrongly and the test stops). Expectation 2 is the test: the predicted gap explains the observed one, with a correlation of at least 0.8 and a slope of observed on predicted between 0.8 and 1.2. Expectation 3 reports, with no pass or fail, the share of market capitalisation outside the industries by calendar month and by year. Expectation 4 asks what remains after the prediction: a tracking error below 25 basis points a year.

In [ ]:
os.makedirs("checks", exist_ok=True)

In [ ]:
%%writefile checks/results_crsp_direct_test.json
{
 "test": "notebook 08, direct test of the explanation of the gap on firm-level data",
 "run_utc": "2026-10-01T14:54Z",
 "firm_level_source": "CRSP monthly stock file and names file, Center for Research in Security Prices, LLC, accessed via WRDS under Queen Mary University of London's subscription on 2026-09-28; data kept on the machine that holds them, results only published",
 "french_vintage": "This file was created using the 202608 CRSP database.",
 "months": 666,
 "first_month": "1969-07",
 "last_month": "2024-12",
 "design": {
  "DIRECT_TEST_SHARE_CODES": [
   10,
   11
  ],
  "DIRECT_TEST_EXCHANGE_CODES": [
   1,
   2,
   3
  ],
  "DIRECT_TEST_FORMATION_MONTH": 6,
  "DIRECT_TEST_MARKET_TE_MAX": 0.002,
  "DIRECT_TEST_MIN_CORRELATION": 0.8,
  "DIRECT_TEST_SLOPE_RANGE": [
   0.8,
   1.2
  ],
  "DIRECT_TEST_RESIDUAL_TE_MAX": 0.0025
 },
 "expectation_1_validation": {
  "months": 666,
  "tracking_error_annual": 0.0007560713421650078,
  "mean_difference_annual": -0.0004227168688761194,
  "correlation": 0.9999888721278982,
  "max_abs_monthly_difference": 0.001535411474569523,
  "month_of_max": "1998-10",
  "level": 0.002,
  "met": true
 },
 "expectation_2_predicted_vs_observed": {
  "months": 666,
  "correlation": 0.9308514413936518,
  "slope": 1.192572977125293,
  "intercept_monthly": -3.406294262470599e-05,
  "te_observed_annual": 0.004733579843260669,
  "te_predicted_annual": 0.003694750513861591,
  "te_residual_annual": 0.0018702672272756153,
  "share_of_variance_explained": 0.8438910100893097,
  "met": true
 },
 "expectation_3_reported": {
  "omega_by_calendar_month_percent": {
   "Jul": 0.57,
   "Aug": 0.759,
   "Sep": 0.879,
   "Oct": 0.988,
   "Nov": 1.11,
   "Dec": 1.277,
   "Jan": 1.685,
   "Feb": 1.798,
   "Mar": 1.951,
   "Apr": 2.085,
   "May": 2.206,
   "Jun": 2.339
  },
  "te_observed_by_calendar_month_bp": {
   "Jul": 2.0,
   "Aug": 6.7,
   "Sep": 11.1,
   "Oct": 14.3,
   "Nov": 34.4,
   "Dec": 37.2,
   "Jan": 27.7,
   "Feb": 54.9,
   "Mar": 86.0,
   "Apr": 80.9,
   "May": 46.3,
   "Jun": 64.8
  },
  "te_predicted_by_calendar_month_bp": {
   "Jul": 6.8,
   "Aug": 12.8,
   "Sep": 10.1,
   "Oct": 17.8,
   "Nov": 27.9,
   "Dec": 29.8,
   "Jan": 19.4,
   "Feb": 39.0,
   "Mar": 71.0,
   "Apr": 61.4,
   "May": 32.4,
   "Jun": 48.4
  },
  "te_residual_by_calendar_month_bp": {
   "Jul": 7.0,
   "Aug": 12.3,
   "Sep": 8.9,
   "Oct": 14.8,
   "Nov": 13.5,
   "Dec": 14.9,
   "Jan": 15.0,
   "Feb": 22.5,
   "Mar": 24.0,
   "Apr": 27.7,
   "May": 22.2,
   "Jun": 27.6
  },
  "omega_by_year_percent_top5": {
   "1973": 8.789,
   "1974": 4.276,
   "1975": 3.787,
   "1976": 3.318,
   "1981": 3.289
  },
  "omega_by_decade_percent": {
   "1960": 0.758,
   "1970": 3.255,
   "1980": 1.965,
   "1990": 1.065,
   "2000": 0.829,
   "2010": 0.706,
   "2020": 0.552
  },
  "firms_outside_industries_by_decade_mean": {
   "1960": 46.0,
   "1970": 605.0,
   "1980": 558.0,
   "1990": 308.0,
   "2000": 135.0,
   "2010": 100.0,
   "2020": 183.0
  },
  "firms_market_by_decade_mean": {
   "1960": 2186.0,
   "1970": 3976.0,
   "1980": 5491.0,
   "1990": 6496.0,
   "2000": 5031.0,
   "2010": 3701.0,
   "2020": 4010.0
  }
 },
 "expectation_4_residual": {
  "te_residual_annual": 0.0018702672272756153,
  "level": 0.0025,
  "met": true
 },
 "largest_observed_gaps": {
  "1973-04": {
   "observed": -0.00542,
   "predicted": -0.00484,
   "omega": 0.131
  },
  "1973-05": {
   "observed": -0.00345,
   "predicted": -0.00316,
   "omega": 0.1261
  },
  "1984-06": {
   "observed": -0.00332,
   "predicted": 0.00027,
   "omega": 0.0642
  },
  "1999-11": {
   "observed": 0.00532,
   "predicted": 0.00472,
   "omega": 0.018
  },
  "1999-12": {
   "observed": 0.00725,
   "predicted": 0.00571,
   "omega": 0.0294
  },
  "2000-02": {
   "observed": 0.00986,
   "predicted": 0.00714,
   "omega": 0.0392
  },
  "2000-03": {
   "observed": -0.0175,
   "predicted": -0.0145,
   "omega": 0.0527
  },
  "2000-04": {
   "observed": -0.01496,
   "predicted": -0.01131,
   "omega": 0.0459
  },
  "2000-05": {
   "observed": -0.00729,
   "predicted": -0.00447,
   "omega": 0.0381
  },
  "2000-06": {
   "observed": 0.01214,
   "predicted": 0.00915,
   "omega": 0.035
  },
  "2000-11": {
   "observed": -0.00392,
   "predicted": -0.00252,
   "omega": 0.0092
  },
  "2001-02": {
   "observed": -0.00424,
   "predicted": -0.00121,
   "omega": 0.0089
  }
 }
}

In [ ]:
import json
R = json.load(open("checks/results_crsp_direct_test.json"))
print(R["firm_level_source"])
print(f"{R['months']} months, {R['first_month']} to {R['last_month']} (the CRSP monthly file on the machine ends in {R['last_month']}); French vintage: {R['french_vintage']}")
e1, e2, e4 = R["expectation_1_validation"], R["expectation_2_predicted_vs_observed"], R["expectation_4_residual"]
print()
print(f"expectation 1: the CRSP market universe against French's market return: tracking error {e1['tracking_error_annual'] * 1e4:.1f} bp a year, "
      f"correlation {e1['correlation']:.5f}, mean difference {e1['mean_difference_annual'] * 1e4:+.1f} bp a year (level {e1['level'] * 1e4:.0f} bp): {'MET' if e1['met'] else 'NOT MET'}")
print(f"expectation 2: correlation between predicted and observed gap {e2['correlation']:.3f} (at least {R['design']['DIRECT_TEST_MIN_CORRELATION']}), "
      f"slope {e2['slope']:.2f} (in {R['design']['DIRECT_TEST_SLOPE_RANGE'][0]} to {R['design']['DIRECT_TEST_SLOPE_RANGE'][1]}): {'MET' if e2['met'] else 'NOT MET'}")
print(f"   tracking error of the observed gap {e2['te_observed_annual'] * 1e4:.1f} bp a year, of the predicted gap {e2['te_predicted_annual'] * 1e4:.1f} bp, "
      f"of what remains {e2['te_residual_annual'] * 1e4:.1f} bp; the prediction accounts for {e2['share_of_variance_explained']:.0%} of the variance of the observed gap")
e3 = R["expectation_3_reported"]
table = pd.DataFrame({"share of market capitalisation outside the industries, %": e3["omega_by_calendar_month_percent"],
                      "tracking error, observed gap, bp": e3["te_observed_by_calendar_month_bp"],
                      "tracking error, predicted gap, bp": e3["te_predicted_by_calendar_month_bp"],
                      "tracking error, what remains, bp": e3["te_residual_by_calendar_month_bp"]})
print()
print("expectation 3, by calendar month, July first:")
print(table.to_string(float_format=lambda v: f"{v:.1f}"))
print()
print("share of market capitalisation outside the industries, by decade, %:", e3["omega_by_decade_percent"])
print("the five years with the largest share, %:", e3["omega_by_year_percent_top5"])
print("firms in the market universe, average per month, by decade:", {k: int(v) for k, v in e3["firms_market_by_decade_mean"].items()})
print("of which outside the industries:", {k: int(v) for k, v in e3["firms_outside_industries_by_decade_mean"].items()})
print()
print(f"expectation 4: what remains, {e4['te_residual_annual'] * 1e4:.1f} bp a year (level {e4['level'] * 1e4:.0f}): {'MET' if e4['met'] else 'NOT MET'}")
print()
print("the twelve months of largest observed gap, with the prediction beside each:")
print(pd.DataFrame(R["largest_observed_gaps"]).T.to_string(float_format=lambda v: f"{v:.4f}"))

## Save the benchmark for the later notebooks

The weights and the two return series are written to `outputs/`, with the vintage in the file name, so that notebooks 09 to 13 can read them or rebuild them from the same module; rebuilding gives the same numbers on the same vintage.

In [ ]:
os.makedirs(C.OUTPUT_DIR, exist_ok=True)
vintage = records[0]["vintage_line"].split("using the ")[1].split(" ")[0]
w_same.to_csv(f"{C.OUTPUT_DIR}/benchmark_weights_{vintage}.csv", float_format="%.8f")
pd.DataFrame({"benchmark": combo_same, "market": market.loc[start:LAST]}).to_csv(f"{C.OUTPUT_DIR}/benchmark_returns_{vintage}.csv", float_format="%.8f")
print("written:", sorted(f for f in os.listdir(C.OUTPUT_DIR) if f.startswith("benchmark")))

## What this notebook established, and what could be wrong

| Expectation | Result | Verdict |
|---|---|---|
| 1. Count first | 686 months from 1969-07 to 2026-08, 49 industries with a return, a firm count and an average size in every month, the market series without a gap | holds |
| 2. The same-month weights track the market more closely than the previous-month weights | 46.7 against 58.7 basis points a year | MET |
| 3. The gate: tracking error at most 50 basis points a year | 46.7 | MET |
| 4. Description of the benchmark | mean difference +9 basis points a year, correlation 0.9996, 93% of months within 0.14%; largest industry 14% in 1969-07, 13% in 2004-11, 22% in 2026-08 | reported |
| Direct test on CRSP, expectations 1 to 4 | the market universe within 7.6 basis points a year of French's market; the predicted gap explains the observed one with correlation 0.93, slope 1.19, 84% of the variance; what remains 18.7 basis points against 25 | MET, 4 of 4 |

Expectation 1 holds. On the 202608 vintage the three blocks share the same 686 months from 1969-07 to 2026-08, all 49 industries have a return, a firm count and an average size in every one of them, and the market series has no gap. The 49 portfolios held 2,152 firms in 1969-07, 4,663 in 2004-11 and 3,170 in 2026-08, with a total market capitalisation that grew from 641 billion dollars to 71 trillion.

Expectation 2 holds. The same-month weights give a tracking error of 46.7 basis points a year against the market and the previous-month weights 58.7, on the 685 months both cover. French's firm count and average firm size for a month therefore describe the start of that month, and the same-month weights carry no look-ahead: a weight for month t is known when month t begins.

Expectation 3, the gate, is met: 46.7 basis points a year over all 686 months against the level of 50 fixed on 10 September. The derived cap weights are the benchmark of notebooks 09 to 13, return series and industry weights alike. Those notebooks rebuild the benchmark from the same module on the same files, so the two CSV files written here are a record and not an input.

Expectation 4, the description. The combination earns 9 basis points a year more than the market on average, the correlation between the two is 0.9996, and in 93% of months they differ by less than 0.14%. The largest monthly difference is 1.75%, in March 2000. The benchmark is concentrated and its concentration moves: the largest industry held 14% in 1969-07 (Oil), 13% in 2004-11 (Banks) and 22% in 2026-08 (Chips); Oil reached 25% in December 1980 and Chips 23% in June 2026; in 2026-08 the eight largest industries hold 68% of the benchmark and 21 of the 49 industries hold less than 0.5% each. This is why the extension's active weight bound is set relative to each industry's benchmark weight (2 percentage points either side of it, `ACTIVE_WEIGHT_BOUND`): one absolute cap for all 49 would be meaningless for an industry at 0.1% and binding for one at 22%.

Where the 47 basis points come from. French forms the industry portfolios once a year, at the end of June, from the firms that have an SIC code at that time, while the market return takes in a new firm from its first full month of trading. A firm that lists in October is in the market from November and in an industry portfolio from the following July, so between July and June the market gains firms the industries do not have, and the gap between the two returns grows. The data show that pattern: the tracking error is 2 basis points a year in July months, 7 in August, 14 in October, 54 in February and 84 in March, and it falls back to 2 the next July. The data also show the other side of the same mechanism: 96 basis points a year in the 2000s, the decade with the most new listings, against 15 in the 2010s. Twelve months around the peak of the technology boom, 1999-07 to 2000-12, account for most of the total; without them the tracking error would be 24 basis points; this number describes the gap and decides nothing. The gate was met with little room, 46.7 against 50. The level was fixed before any of this was computed and is not revisited after the fact; had the level been 40, the derived weights would have been rejected and the benchmark return would be the market series.

The direct test confirms the explanation. On the CRSP monthly stock file, 666 months from 1969-07 to 2024-12, the market universe built from share codes 10 and 11 on the three exchanges reproduces French's market return to within 7.6 basis points a year (correlation 0.99999), so the universes are built the way French builds his. The gap that the June-formed universe predicts has a correlation of 0.93 with the observed gap, a slope of 1.19, and it accounts for 84% of the observed gap's variance; what remains has a tracking error of 18.7 basis points a year against the level of 25. All four expectations are met. The share of market capitalisation held by firms outside the industries rises from 0.6% in July months to 2.3% in June months, the shape the formation rule implies. Its largest values are in 1973, when CRSP added the NASDAQ stocks to its database: the market universe went from 2,505 firms in December 1972 to 5,364 in January 1973, and the 2,960 new firms, 13% of market capitalisation, were in no industry portfolio until July 1973; the two largest gaps of the 1970s, April and May 1973, are reproduced by the prediction to within 0.06 of a percentage point. In March 2000 the firms outside the industries held 5.3% of market capitalisation and the prediction gives -1.45% against the observed -1.75%.

**What this notebook does not settle.**

- What remains after the prediction, 18.7 basis points a year, and the slope above one: the observed gap is about a fifth larger than the predicted one. Three differences between the test's universes and French's construction can account for it, and the test does not separate them. French assigns industries by Compustat's SIC code where one exists and by CRSP's only otherwise, while the test uses CRSP's alone, so a firm with a CRSP code of zero and a Compustat code is outside the industries in the test and inside them for French; in July 1973, 677 of the 5,161 market firms had no CRSP SIC code. French's returns include the return a firm earns in the month it is delisted, which the CRSP monthly file carries in a separate table the test does not read. And French's average firm size is rounded to two decimals of a million dollars. Separating the three would need Compustat and the delisting table, which the study does not use.
- The CRSP file on the machine ends in December 2024, so the last twenty months of the 49-industry combination are outside the test.
- The level of 50 basis points was a judgement about how close to the market an index of the 49 industries has to be to stand in for it; nothing in the extension depends on that closeness, because every later tracking error is measured against this benchmark, so the 47 basis points do not enter any later number.
- A cap-weighted index rebalances itself: when an industry's prices rise, its weight rises with them and no trade is needed. The portfolios of notebooks 11 and 12 have to trade to follow it, a limit on turnover (the fraction of the portfolio traded each month) caps how much they may trade, and notebook 12 measures what that costs.